# Agent

Build agents on the **PhysiCar AI Services**. Two agents — **chat** (text, `POST /chat`) and **realtime** (voice, `wss://.../realtime`) — sharing **one brain**: the instructions and the robot tools in `assets/agent/web/prompt.js`. Everything runs in the browser — Python only serves the pages.

Auth comes from the signed-in PhysiCar session: on any `/myapp/` page, `physicarSession.token()` is auto-injected by nginx (sign in once on the App page). Chat sends it as `Authorization: Bearer <token>`; realtime as the WebSocket subprotocol `token.<token>`. Never cache one token globally on a shared robot — read it per request so every user stays on their own account.

Models differ per service: chat models come from `GET /chat/models` (text LLMs); the realtime service runs its own speech-to-speech model, assigned by the service.

## [0] The shared brain — instructions & tools

### 0.1. The system prompt

Edit it **here** and re-run — the Run cells below inject it into the page, so a reload of the MYAPP tab picks it up. (The same text in `assets/agent/web/prompt.js` is only the fallback when the pages are served without this notebook.)

Every rule exists for a reason: the 0.5 m/s default keeps demos safe, `duration` rides the web API's blocking drive (the response returns when the car stops, so the model can chain actions), and camera-then-answer forces the model to actually look.

In [ ]:
INSTRUCTIONS = """
You are PhysiCar, a self-driving robot. Respond in the user's language, short and friendly.
When asked to act, you MUST call a tool — never just say you will and skip the call.
Drive gently — use speed 0.5 m/s by default; go faster only if the user insists.
To actually go somewhere, always pass duration (seconds) — a speed without
duration expires after ~1 s (safety watchdog) and the car stops by itself.
When told to stop, immediately call drive(speed=0, steering=0).
"What do you see?" -> call camera first, then answer. "Anything around me?" -> read lidar.
For music: find it with music_search, then play with music_player(action=play).
"""

### 0.2. Tools

Each tool is `{description, properties, run}`; the description and parameter schema go to the model — **for both agents, in the same shape** — and `run(args)` executes in the browser when the model calls it. The drive tool, verbatim:

```js
drive: run: async ({speed = 0, steering = 0, duration}) => {
  await fetch('/steering', {method: 'POST',
    headers: {'Content-Type': 'application/json'},
    body: JSON.stringify({value: steering * Math.PI / 180})});  // API wants radians
  const body = {value: speed};
  if (duration) body.duration = Math.max(1, Math.min(120, duration));
  // With duration this response blocks until the car has stopped —
  // chain the next action directly, no sleep needed.
  await fetch('/speed', {method: 'POST',
    headers: {'Content-Type': 'application/json'},
    body: JSON.stringify(body)});
  return `speed ${speed} m/s, steering ${steering} deg`
    + (duration ? `, drove ${body.duration} s and stopped` : '');
}
```

Shipped tools: `drive`, `look` (camera pan/tilt), `sleep`, `camera` (returns the photo INTO the conversation so the model describes it), `lidar` (`/lidar?step=deg`, 0°=front, +90°=left), `states`, `music_search` (iTunes), `music_player` (`/audio/play` on the robot speaker).

To add a tool: copy an entry in `TOOLS`, give it a clear `description` + `properties`, return a string from `run` — both agents discover it on their next session/turn.

## [1] Chat Agent

### 1.1. The protocol

One request per turn: the prompt (model + instructions + tools) is sent **inline on every request** — the server keeps only the conversation (`chat_id`/`turn`, retained 24 h). The response is an SSE stream. Pick a model from `GET /chat/models`.

```js
const res = await fetch("https://api.physicar.ai/chat", {
  method: "POST",
  headers: { "Content-Type": "application/json",
             "Authorization": "Bearer " + physicarSession.token() },
  body: JSON.stringify({
    chat_id: chatId,            // undefined on the first turn
    turn,                       // 0 on the first turn
    prompt: { model, instructions: INSTRUCTIONS, tools: toolDefs },
    user_message: { contents: [{ type: "text", text: "Hello!" }] },
    stream: true,
  }),
});
```

SSE frames (`data: {...}` JSON, one per event):

| Frame | Meaning |
|-------|---------|
| `{type:"text", content}` | streamed answer text |
| `{type:"done", chat_id, turn, tool_calls, finish_reason, usage}` | turn finished — keep `chat_id` and send `turn + 1` next time |
| `{type:"error", message}` | stream error |

**Tool loop**: when `done.tool_calls` is non-empty, run each `{call_id, name, arguments}` with the shared `TOOLS[name].run` and send the results back as the next request's user message:

```js
user_message: { contents: [], tool_call_outputs: [
  { call_id, name, contents: [{ type: "text", text: result }] },
] }
```

The PhysiCar Chat panel in VS Code is exactly this API. The full working page is `assets/agent/web/chat.html` — a ~200-line messenger UI implementing this protocol on the shared brain.

### 1.2. Run

Serve the chat page, then open the **MYAPP** tab in the App page and talk by text — ask it to drive, look around, describe what it sees (tool calls show as ⚙ lines, camera photos appear inline). The cell keeps running while the server lives — **interrupt it (⏹) to stop**. (Leaving the tab stops the car: a `sendBeacon` zeroes speed/steering on page hide.)

In [ ]:
import json
import os

MODEL = ""      # chat model id — empty = the first from GET /chat/models

DIR = "assets/agent"
if not os.path.isdir(DIR):               # kernel cwd is the workspace root
    DIR = "examples/" + DIR

import socket

probe = socket.socket()
probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
try:
    probe.bind(("", 5000))
    probe.close()
except OSError:
    raise SystemExit("port 5000 is in use — another example is serving on the MYAPP tab; "
                     "interrupt (⏹) its running cell (or restart that kernel), then re-run")

from flask import Flask, send_from_directory

WEB = os.path.abspath(f"{DIR}/web")
app = Flask(__name__)


@app.route("/")
def index():
    # inject the notebook's prompt (and model) into the page
    page = open(f"{WEB}/chat.html", encoding="utf-8").read()
    page = page.replace('"__INSTRUCTIONS__"', json.dumps(INSTRUCTIONS))
    page = page.replace('"__MODEL__"', json.dumps(MODEL))
    return page


@app.route("/<path:name>")
def static_file(name):
    return send_from_directory(WEB, name)


print("serving on port 5000 — open the MYAPP tab in the App page")
app.run(host="0.0.0.0", port=5000)

## [2] Realtime Agent

### 2.1. The protocol

The speech-to-speech service:

```
mic ──▶ realtime cloud (speech-to-speech LLM) ──▶ speaker
              │            ▲
         tool_call    tool_result
              ▼            │
        JS tools in the page ──▶ robot web API (/speed, /camera, ...)
```

The page opens `wss://api.physicar.ai/realtime` with subprotocol `token.<token>`, sends `{type:'session.start', prompt:{instructions: INSTRUCTIONS, tools: toolDefs}}` — **the same shared brain** — streams mic audio in (`{type:'audio', data}` PCM16 base64) and receives `audio.delta` / `transcript.delta` / `tool_call` / `interrupted` / `session.end` events. Tool results go back as `{type:'tool_result', call_id, name, output}`.

The full working page is `assets/agent/web/index.html` (plumbing — mic, speaker, reconnect; no need to read it).

### 2.2. Run

Serve the voice page, then open the **MYAPP** tab in the App page. The mic starts on — just talk. The cell keeps running while the server lives — **interrupt it (⏹) to stop**. (Closing the tab stops the car the same way.)

In [ ]:
import json
import os

DIR = "assets/agent"
if not os.path.isdir(DIR):               # kernel cwd is the workspace root
    DIR = "examples/" + DIR

import socket

probe = socket.socket()
probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
try:
    probe.bind(("", 5000))
    probe.close()
except OSError:
    raise SystemExit("port 5000 is in use — another example is serving on the MYAPP tab; "
                     "interrupt (⏹) its running cell (or restart that kernel), then re-run")

from flask import Flask, send_from_directory

WEB = os.path.abspath(f"{DIR}/web")
app = Flask(__name__)


@app.route("/")
def index():
    # inject the notebook's prompt (and model) into the page
    page = open(f"{WEB}/index.html", encoding="utf-8").read()
    page = page.replace('"__INSTRUCTIONS__"', json.dumps(INSTRUCTIONS))
    return page


@app.route("/<path:name>")
def static_file(name):
    return send_from_directory(WEB, name)


print("serving on port 5000 — open the MYAPP tab in the App page")
app.run(host="0.0.0.0", port=5000)